# Combined ML + Strategy Predictions

Ensemble approach combining LSTM predictions and scalping strategy signals for improved trading decisions.

In [27]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path.cwd().parent))

from src.data_collection.load_kaggle_data import load_kaggle_data
from src.preprocessing.clean_data import clean_ohlcv_data
from src.utils.data_split import split_data_by_date
from src.utils.config import DEFAULT_TICKERS, TRAIN_START, TRAIN_END, TEST_START, TEST_END

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import joblib

print("="*80)
print("COMBINED ML + STRATEGY BACKTESTING")
print("="*80)
print(f"Testing Period: {TEST_START} to {TEST_END}")
print("="*80)

COMBINED ML + STRATEGY BACKTESTING
Testing Period: 2024-01-01 to 2024-12-31


## Define Feature Engineering and Scalping Strategy

Replicate feature engineering and strategy logic from earlier notebooks.

In [28]:
# ======================================================
# SCALPING STRATEGY SIGNALS (Rule-Based)
# ======================================================
def add_scalping_signals(data):
    df = data.copy()

    # RSI
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    rsi = 100 - (100 / (1 + rs))

    # Moving averages
    sma_20 = df["Close"].rolling(20).mean()
    sma_50 = df["Close"].rolling(50).mean()

    # MACD
    ema_12 = df["Close"].ewm(span=12).mean()
    ema_26 = df["Close"].ewm(span=26).mean()
    macd = ema_12 - ema_26
    macd_signal = macd.ewm(span=9).mean()
    macd_hist = macd - macd_signal

    # BUY conditions
    buy_uptrend = (df["Close"] > sma_20) & (sma_20 > sma_50)
    buy_rsi = rsi < 40
    buy_macd = (macd > 0) & (macd_hist > 0)

    close_20_high = df["Close"].rolling(20).max()
    buy_strength = df["Close"] > 0.95 * close_20_high

    buy_signal = (
        (buy_uptrend & buy_rsi) |
        (buy_uptrend & buy_macd) |
        (buy_uptrend & buy_strength)
    )

    # SELL conditions
    sell_downtrend = (df["Close"] < sma_20) | (sma_20 < sma_50)
    sell_rsi = rsi > 60
    sell_macd = (macd < 0) & (macd_hist < 0)

    sell_signal = (
        (sell_downtrend & sell_rsi) |
        (sell_downtrend & sell_macd)
    )

    # Final signal
    signal = pd.Series(0, index=df.index)
    signal[buy_signal] = 1
    signal[sell_signal] = -1
    signal[(buy_signal) & (sell_signal)] = 1

    df["strategy_signal"] = signal
    return df


# ======================================================
# FEATURE ENGINEERING (ML + Trading Aligned)
# ======================================================
def add_basic_features(data, horizon=3, cost=0.0003):
    df = data.copy()

    # Returns
    df["returns"] = df["Close"].pct_change()
    df["log_returns"] = np.log(df["Close"] / df["Close"].shift(1))

    # Trend
    sma_10 = df["Close"].rolling(10).mean()
    sma_20 = df["Close"].rolling(20).mean()

    df["trend_10"] = (df["Close"] - sma_10) / sma_10
    df["trend_20"] = (df["Close"] - sma_20) / sma_20
    df["trend_diff"] = (sma_10 - sma_20) / sma_20

    # Price action
    df["range_pct"] = (df["High"] - df["Low"]) / df["Close"]
    df["body_pct"] = (df["Close"] - df["Open"]) / df["Close"]
    df["body_abs"] = df["body_pct"].abs()

    # Volatility regime
    df["volatility_10"] = df["returns"].rolling(10).std()
    df["vol_ratio"] = df["volatility_10"] / df["volatility_10"].rolling(50).mean()
    df["high_vol"] = (df["vol_ratio"] > 1.0).astype(int)

    # RSI (0–1)
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    df["RSI"] = (100 - (100 / (1 + rs))) / 100.0

    # Volume
    if "Volume" in df.columns and df["Volume"].sum() > 0:
        vol_sma = df["Volume"].rolling(20).mean()
        df["Volume_norm"] = np.log1p(df["Volume"] / (vol_sma + 1e-8))
    else:
        df["Volume_norm"] = 0.0

    # Target: forward return beyond cost
    future_return = (df["Close"].shift(-horizon) - df["Close"]) / df["Close"]
    df["target"] = (future_return > cost).astype(int)

    df.dropna(inplace=True)
    return df






## Load Data and Generate Both ML + Strategy Predictions

For first ticker, compare:
1. ML predictions (LSTM only)
2. Strategy signals (technical rules only)
3. Combined predictions (voting ensemble)

In [29]:
ticker = DEFAULT_TICKERS[0]
print(f"\n{'='*80}")
print(f"ANALYZING {ticker}")
print(f"{'='*80}")

# Load and prepare data
raw_data = load_kaggle_data(ticker)
cleaned_data = clean_ohlcv_data(raw_data)
train_data, test_data = split_data_by_date(cleaned_data)

print(f"Train data: {train_data.shape}")
print(f"Test data: {test_data.shape}")

# Feature engineering for ML
# ------------------------------------------------------
# Apply strategy FIRST (raw data)
# ------------------------------------------------------
train_with_signals = add_scalping_signals(train_data)
test_with_signals  = add_scalping_signals(test_data)

# ------------------------------------------------------
# Then apply feature engineering (keeps alignment)
# ------------------------------------------------------
train_with_features = add_basic_features(train_with_signals)
test_with_features  = add_basic_features(test_with_signals)

print(f"Train with features: {train_with_features.shape}")
print(f"Test with features:  {test_with_features.shape}")



ANALYZING NIFTY BANK
2025-12-25 22:06:09 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Sunay Bhattacharjee\Desktop\AlgoTrading bot project\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2025-12-25 22:06:11 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2025-12-25 22:06:12 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2025-12-25 22:06:12 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2025-12-25 22:06:12 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2025-12-25 22:06:12 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2025-12-25 22:06:12 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2025-12-25 22:06:12 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-01-09 09:15:00 to 2025-04-16 0

In [30]:
# ======================================================
# STEP 1: ML PREDICTIONS (XGBoost - IMPROVED)
# ======================================================
print("\n" + "="*80)
print("STEP 1: XGBoost ML MODEL PREDICTIONS (IMPROVED)")
print("="*80)

# ------------------------------------------------------
# Feature selection
# ------------------------------------------------------
feature_cols = [
    col for col in train_with_features.columns
    if col not in ['target', 'Open', 'High', 'Low', 'Close', 'Volume']
]

X_train = train_with_features[feature_cols]
y_train = train_with_features['target']

X_test = test_with_features[feature_cols]
y_test = test_with_features['target']

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

# ------------------------------------------------------
# Time-based validation split (NO leakage)
# ------------------------------------------------------
val_size = int(0.8 * len(X_train))
X_tr, X_val = X_train.iloc[:val_size], X_train.iloc[val_size:]
y_tr, y_val = y_train.iloc[:val_size], y_train.iloc[val_size:]

# ------------------------------------------------------
# XGBoost model (tuned for noisy financial data)
# ------------------------------------------------------
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=20,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    verbosity=0
)

print("\nTraining XGBoost model...")

xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)



# ------------------------------------------------------
# Probabilities
# ------------------------------------------------------
y_val_prob = xgb_model.predict_proba(X_val)[:, 1]
y_test_prob = xgb_model.predict_proba(X_test)[:, 1]

# ------------------------------------------------------
# Threshold optimization (EDGE-BASED, validation only)
# ------------------------------------------------------
best_threshold = 0.5
best_score = -np.inf

baseline = y_val.mean()   # base probability of positive target

for t in np.arange(0.35, 0.65, 0.02):
    preds = (y_val_prob > t).astype(int)
    trade_rate = preds.mean()

    # avoid extreme over/under trading
    if trade_rate < 0.05 or trade_rate > 0.4:
        continue

    if preds.sum() > 0:
        win_rate = y_val[preds == 1].mean()
    else:
        win_rate = 0.0

    edge = win_rate - baseline

    # penalize tiny sample sizes
    score = edge * np.sqrt(preds.sum())

    if score > best_score:
        best_score = score
        best_threshold = t


# ------------------------------------------------------
# Final test predictions
# ------------------------------------------------------
y_test_pred = (y_test_prob > best_threshold).astype(int)

ml_accuracy = accuracy_score(y_test, y_test_pred)
ml_auc = roc_auc_score(y_test, y_test_prob)
ml_f1 = f1_score(y_test, y_test_pred, zero_division=0)

print(f"\n✓ ML Threshold: {best_threshold:.2f}")
print(f"✓ ML Test Accuracy: {ml_accuracy:.4f}")
print(f"✓ ML Test AUC:      {ml_auc:.4f}")
print(f"✓ ML Test F1:       {ml_f1:.4f}")



STEP 1: XGBoost ML MODEL PREDICTIONS (IMPROVED)
X_train: (791543, 14)
X_test:  (80848, 14)
y_train: (791543,)
y_test:  (80848,)

Training XGBoost model...

✓ ML Threshold: 0.35
✓ ML Test Accuracy: 0.6588
✓ ML Test AUC:      0.6041
✓ ML Test F1:       0.3402


In [31]:
test_eval = test_with_features.copy()

test_eval["ml_prob"] = y_test_prob[:len(test_eval)]
test_eval["ml_entry"] = (test_eval["ml_prob"] > best_threshold).astype(int)

test_eval["strategy_entry"] = (test_eval["strategy_signal"] == 1).astype(int)

test_eval["combined_entry"] = (
    (test_eval["strategy_entry"] == 1) &
    (test_eval["ml_entry"] == 1)
).astype(int)

baseline = y_test[:len(test_eval)].mean()

def evaluate(name, col):
    entries = test_eval[col]
    trade_rate = entries.mean()
    win_rate = y_test[:len(test_eval)][entries == 1].mean() if entries.sum() > 0 else 0
    edge = win_rate - baseline
    print(f"\n--- {name} ---")
    print(f"Trade rate: {trade_rate:.2%}")
    print(f"Win rate:   {win_rate:.2%}")
    print(f"Edge:       {edge:.2%}")

evaluate("STRATEGY ONLY", "strategy_entry")
evaluate("ML ONLY", "ml_entry")
evaluate("STRATEGY + ML", "combined_entry")



--- STRATEGY ONLY ---
Trade rate: 30.47%
Win rate:   26.42%
Edge:       -1.74%

--- ML ONLY ---
Trade rate: 23.55%
Win rate:   37.35%
Edge:       9.19%

--- STRATEGY + ML ---
Trade rate: 4.92%
Win rate:   36.13%
Edge:       7.97%


In [32]:
# ======================================================
# STEP 2: STRATEGY AS CONTEXT FILTER (ML-ALIGNED)
# ======================================================
print("\n" + "="*80)
print("STEP 2: STRATEGY AS CONTEXT FILTER (ML-ALIGNED)")
print("="*80)

# Base evaluation frame (must already be aligned)
test_eval = test_with_features.copy()

# ------------------------------------------------------
# Entries
# ------------------------------------------------------
test_eval["strategy_entry"] = (test_eval["strategy_signal"] == 1).astype(int)
test_eval["ml_entry"] = (y_test_prob > best_threshold).astype(int)

test_eval["combined_entry"] = (
    (test_eval["strategy_entry"] == 1) &
    (test_eval["ml_entry"] == 1)
).astype(int)

# Align target
y_target = y_test.values[:len(test_eval)]
baseline = y_target.mean()

# ------------------------------------------------------
# Evaluation helper
# ------------------------------------------------------
def evaluate(name, entry_col):
    entries = test_eval[entry_col]
    trade_rate = entries.mean()
    
    if entries.sum() > 0:
        win_rate = y_target[entries == 1].mean()
    else:
        win_rate = 0.0

    edge = win_rate - baseline

    print(f"\n--- {name} ---")
    print(f"Trade rate: {trade_rate:.2%}")
    print(f"Win rate:   {win_rate:.2%}")
    print(f"Edge:       {edge:.2%}")

# ------------------------------------------------------
# Results
# ------------------------------------------------------
evaluate("STRATEGY ONLY", "strategy_entry")
evaluate("ML ONLY", "ml_entry")
evaluate("STRATEGY + ML", "combined_entry")



STEP 2: STRATEGY AS CONTEXT FILTER (ML-ALIGNED)

--- STRATEGY ONLY ---
Trade rate: 30.47%
Win rate:   26.42%
Edge:       -1.74%

--- ML ONLY ---
Trade rate: 23.55%
Win rate:   37.35%
Edge:       9.19%

--- STRATEGY + ML ---
Trade rate: 4.92%
Win rate:   36.13%
Edge:       7.97%


## Key Findings

Summary of the combined ML + Strategy approach compared to individual techniques.

In [33]:
# =====================================================
# BACKTEST CONFIG (MAX-RETURN READY)
# =====================================================

INITIAL_CAPITAL = 1_000_000
RISK_FREE_RATE = 0.0

HORIZON =60

# -------------------------------
# ML ENTRY (TAIL ONLY)
# -------------------------------
ENTRY_Q = 0.96    # trade only top 5% signals

# -------------------------------
# EXECUTION CONTROLS
# -------------------------------
STOP_LOSS = 0.004
COST_PER_TRADE = 0.00015

# Position sizing (convex)
SIZE_EXPONENT = 3
MAX_POSITION = 1.0



# Trade management
COOLDOWN = HORIZON//2


In [34]:
ticker = "NIFTY BANK"
print(f"\nBacktesting: {ticker}")

# --------------------------------------------------
# Load & clean
# --------------------------------------------------
raw = load_kaggle_data(ticker)
cleaned = clean_ohlcv_data(raw)
train_data, test_data = split_data_by_date(cleaned)

# --------------------------------------------------
# CONTINUOUS FEATURE PIPELINE (CRITICAL FIX)
# --------------------------------------------------
# Combine train + test to preserve rolling context
full_data = pd.concat([train_data, test_data], axis=0)

# Apply features on full history
full_with_signals = add_scalping_signals(full_data)
full_features = add_basic_features(full_with_signals)

# Slice back test portion ONLY
test_df = full_features.loc[test_data.index]

# --------------------------------------------------
# ML inputs
# --------------------------------------------------
feature_cols = [
    c for c in test_df.columns
    if c not in ["target", "Open", "High", "Low", "Close", "Volume"]
]

X_test = test_df[feature_cols]
y_test = test_df["target"]
prices = test_df["Close"].values



Backtesting: NIFTY BANK
2025-12-25 22:06:57 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Sunay Bhattacharjee\Desktop\AlgoTrading bot project\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2025-12-25 22:06:59 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2025-12-25 22:06:59 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2025-12-25 22:06:59 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2025-12-25 22:06:59 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2025-12-25 22:06:59 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2025-12-25 22:06:59 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2025-12-25 22:06:59 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-01-09 09:15:00 to 2025-04-1

In [35]:
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------
# TRAIN FEATURES (DEFINE FEATURE SPACE HERE)
# --------------------------------------------------
train_with_signals = add_scalping_signals(train_data)
train_df = add_basic_features(train_with_signals)

feature_cols = [
    c for c in train_df.columns
    if c not in ["target", "Open", "High", "Low", "Close", "Volume"]
]

X_train = train_df[feature_cols]
y_train = train_df["target"]

# --------------------------------------------------
# SCALE (FIT ON TRAIN ONLY)
# --------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --------------------------------------------------
# MODEL
# --------------------------------------------------
model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=20,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    verbosity=0
)

model.fit(X_train_scaled, y_train)

# --------------------------------------------------
# ML PROBABILITIES
# --------------------------------------------------
ml_prob = model.predict_proba(X_test_scaled)[:, 1]
print("Test AUC:", roc_auc_score(y_test, ml_prob))


Test AUC: 0.6049132125626437


In [36]:
def position_size(prob, threshold):
    """
    Convert ML probability into position size [0, 1]
    """
    size = (prob - threshold) / (1 - threshold)
    return np.clip(size, 0, 1)


In [37]:

capital = INITIAL_CAPITAL
equity_curve = []
trades = []

# ---------------------------------
# ENTRY THRESHOLD (TAIL ONLY)
# ---------------------------------
ENTRY_THRESHOLD = np.quantile(ml_prob, ENTRY_Q)

i = 0
n = len(prices)

while i < n - HORIZON:

    prob = ml_prob[i]

    # --------------------------------------------------
    # 1. SKIP LOW-CONFIDENCE SIGNALS
    # --------------------------------------------------
    if prob <= ENTRY_THRESHOLD:
        equity_curve.append(capital)
        i += 1
        continue

    # --------------------------------------------------
    # 2. VOLATILITY-BASED RISK (KNOWN AT ENTRY)
    # --------------------------------------------------
    vol = test_df["volatility_10"].iloc[i]

    STOP_LOSS = 1.2 * vol        # adaptive risk
    MAX_HOLD  = HORIZON          # hard time cap

    # --------------------------------------------------
    # 3. POSITION SIZING
    # --------------------------------------------------
    edge_strength = (prob - ENTRY_THRESHOLD) / (1 - ENTRY_THRESHOLD)
    size = np.clip(edge_strength ** SIZE_EXPONENT, 0.25, 1.0)

    risk_adj = np.clip(0.01 / vol, 0.5, 2.0)

    position_value = capital * size * risk_adj
    entry_price = prices[i]

    # --------------------------------------------------
    # 4. EXIT LOGIC (STOP LOSS + TIME ONLY)
    # --------------------------------------------------
    exit_price = prices[i + MAX_HOLD]
    exit_idx   = i + MAX_HOLD

    for j in range(1, MAX_HOLD + 1):
        price = prices[i + j]

        # STOP LOSS ONLY
        if price <= entry_price * (1 - STOP_LOSS):
            exit_price = entry_price * (1 - STOP_LOSS)
            exit_idx = i + j
            break

    # --------------------------------------------------
    # 5. PnL CALCULATION (ONCE)
    # --------------------------------------------------
    ret = (exit_price - entry_price) / entry_price
    net_ret = ret - COST_PER_TRADE

    pnl = position_value * net_ret
    capital += pnl

    trades.append({
        "entry_idx": i,
        "exit_idx": exit_idx,
        "prob": prob,
        "size": size,
        "return": net_ret,
        "pnl": pnl,
        "capital": capital
    })

    equity_curve.append(capital)

    # --------------------------------------------------
    # 6. COOLDOWN
    # --------------------------------------------------
    i += COOLDOWN


In [38]:
equity = pd.Series(equity_curve)
returns = equity.pct_change().dropna()

total_return = (equity.iloc[-1] / equity.iloc[0]) - 1
max_dd = ((equity / equity.cummax()) - 1).min()

sharpe = (
    returns.mean() / returns.std()
    if returns.std() > 0 else 0
) * np.sqrt(252 * 6.5 * 60)   # intraday annualization

profit_factor = (
    sum(t["pnl"] for t in trades if t["pnl"] > 0) /
    abs(sum(t["pnl"] for t in trades if t["pnl"] < 0))
    if any(t["pnl"] < 0 for t in trades) else np.inf
)

print("\n================ BACKTEST SUMMARY ================")
print(f"Final Capital:     ₹{capital:,.0f}")
print(f"Total Return:      {total_return*100:.2f}%")
print(f"Max Drawdown:      {max_dd*100:.2f}%")
print(f"Sharpe Ratio:      {sharpe:.2f}")
print(f"Profit Factor:    {profit_factor:.2f}")
print(f"Total Trades:     {len(trades)}")



================ BACKTEST SUMMARY ================
Final Capital:     ₹1,176,320
Total Return:      17.53%
Max Drawdown:      -1.37%
Sharpe Ratio:      4.80
Profit Factor:    1.77
Total Trades:     558


In [39]:
trade_returns = [t["return"] for t in trades]

print("\nTrade Stats")
print(f"Win rate: {np.mean(np.array(trade_returns) > 0)*100:.2f}%")
print(f"Avg win:  {np.mean([r for r in trade_returns if r > 0])*100:.2f}%")
print(f"Avg loss: {np.mean([r for r in trade_returns if r < 0])*100:.2f}%")



Trade Stats
Win rate: 30.11%
Avg win:  0.45%
Avg loss: -0.11%


In [ ]:
# ======================================================
# LIVE PAPER TRADING — 1 MIN INTRADAY (NIFTY BANK)
# FULLY ALIGNED WITH BACKTEST
# ======================================================

import yfinance as yf
import time
from datetime import datetime, time as dtime
import pandas as pd

SYMBOL = "^NSEBANK"
INTERVAL = "1m"

# ------------------------------------------------------
# RESET RUNTIME STATE (do NOT reuse backtest variables)
# ------------------------------------------------------
paper_capital = INITIAL_CAPITAL
position = 0
entry_price = None
entry_time = None
qty = 0
invested_amount = 0

paper_trades = []

MARKET_OPEN = dtime(9, 15)
MARKET_CLOSE = dtime(15, 15)

# ------------------------------------------------------
# DATA FETCH
# ------------------------------------------------------
def fetch_1min_data():
    df = yf.download(
        SYMBOL,
        period="2d",
        interval="1m",
        progress=False
    )
    return df.dropna()

print("=" * 80)
print("STARTING 1-MIN INTRADAY PAPER TRADING — NIFTY BANK")
print("=" * 80)

# ------------------------------------------------------
# LIVE LOOP
# ------------------------------------------------------
while True:

    now = datetime.now().time()

    # Wait for market open
    if now < MARKET_OPEN:
        time.sleep(20)
        continue

    # Market close handling
    if now > MARKET_CLOSE:
        if position == 1:
            df = fetch_1min_data()
            price = df["Close"].iloc[-1]

            pnl_cash = (price - entry_price) * qty
            paper_capital += pnl_cash - invested_amount * COST_PER_TRADE

            paper_trades.append({
                "time": datetime.now(),
                "side": "FORCED_SELL",
                "price": price,
                "pnl_cash": pnl_cash,
                "capital": paper_capital
            })

        print("MARKET CLOSED — STOPPING PAPER TRADING")
        break

    try:
        # Fetch data
        df = fetch_1min_data()
        df_feat = add_scalping_signals(df)
        latest = df_feat.iloc[-1:]

        # ML inference
        X_live = latest[feature_cols]
        X_live_scaled = scaler.transform(X_live)
        prob = model.predict_proba(X_live_scaled)[0, 1]

        price = df["Close"].iloc[-1]
        ts = df.index[-1]

        # --------------------------------------------------
        # ENTRY (matches backtest)
        # --------------------------------------------------
        if position == 0 and prob >= ENTRY_Q:
            invested_amount = paper_capital * RISK_PER_TRADE
            qty = invested_amount / price

            position = 1
            entry_price = price
            entry_time = ts

            paper_trades.append({
                "time": ts,
                "side": "BUY",
                "price": price,
                "prob": prob,
                "capital": paper_capital
            })

            print(f"[BUY] {ts} | Price={price:.2f} | Prob={prob:.3f}")

        # --------------------------------------------------
        # EXIT (matches backtest)
        # --------------------------------------------------
        if position == 1:
            pnl_pct = (price - entry_price) / entry_price
            hold_minutes = (ts - entry_time).seconds // 60

            if (
                pnl_pct <= -STOP_LOSS or
                pnl_pct >= STOP_LOSS * 1.5 or
                hold_minutes >= HORIZON
            ):
                pnl_cash = (price - entry_price) * qty
                paper_capital += pnl_cash - invested_amount * COST_PER_TRADE

                paper_trades.append({
                    "time": ts,
                    "side": "SELL",
                    "price": price,
                    "pnl_cash": pnl_cash,
                    "capital": paper_capital
                })

                print(
                    f"[SELL] {ts} | PnL={pnl_pct*100:.2f}% "
                    f"| Capital={paper_capital:,.0f}"
                )

                position = 0
                entry_price = None
                entry_time = None
                qty = 0
                invested_amount = 0

        # Sync to candle close
        sleep_seconds = 60 - datetime.now().second
        time.sleep(max(sleep_seconds, 1))

    except Exception as e:
        print("ERROR:", e)
        time.sleep(30)


STARTING 1-MIN INTRADAY PAPER TRADING — NIFTY BANK
MARKET CLOSED — STOPPING PAPER TRADING


In [56]:
# ======================================================
# PAPER TRADING PERFORMANCE ANALYSIS
# ======================================================

import pandas as pd
import numpy as np

paper_df = pd.DataFrame(paper_trades)

if paper_df.empty:
    print("No trades executed.")
else:
    # -------------------------------
    # Separate BUY / SELL trades
    # -------------------------------
    buys = paper_df[paper_df["side"] == "BUY"].reset_index(drop=True)
    sells = paper_df[paper_df["side"].isin(["SELL", "FORCED_SELL"])].reset_index(drop=True)

    trades = pd.concat([buys, sells], axis=1)
    trades = trades.loc[:, ~trades.columns.duplicated()]

    # -------------------------------
    # Returns
    # -------------------------------
    trades["return"] = trades["pnl"]
    trades.dropna(inplace=True)

    total_trades = len(trades)
    wins = trades[trades["return"] > 0]
    losses = trades[trades["return"] <= 0]

    win_rate = len(wins) / total_trades if total_trades > 0 else 0
    avg_win = wins["return"].mean() if not wins.empty else 0
    avg_loss = losses["return"].mean() if not losses.empty else 0

    profit_factor = (
        wins["return"].sum() / abs(losses["return"].sum())
        if not losses.empty else np.inf
    )

    # -------------------------------
    # Equity Curve & Drawdown
    # -------------------------------
    equity = paper_df["capital"].dropna()
    peak = equity.cummax()
    drawdown = (equity - peak) / peak

    max_dd = drawdown.min()

    # -------------------------------
    # Sharpe (intraday approx)
    # -------------------------------
    returns = trades["return"]
    sharpe = (
        np.sqrt(252 * 6.5 * 60) * returns.mean() / returns.std()
        if returns.std() != 0 else 0
    )

    # -------------------------------
    # Print Summary
    # -------------------------------
    print("=" * 60)
    print("INTRADAY PAPER TRADING SUMMARY")
    print("=" * 60)
    print(f"Final Capital      : ₹{paper_capital:,.0f}")
    print(f"Total Trades       : {total_trades}")
    print(f"Win Rate           : {win_rate*100:.2f}%")
    print(f"Avg Win            : {avg_win*100:.3f}%")
    print(f"Avg Loss           : {avg_loss*100:.3f}%")
    print(f"Profit Factor      : {profit_factor:.2f}")
    print(f"Max Drawdown       : {max_dd*100:.2f}%")
    print(f"Sharpe Ratio       : {sharpe:.2f}")
    print("=" * 60)

    trades


No trades executed.


In [54]:
paper_df = pd.DataFrame(paper_trades)
paper_df


""


In [57]:
import os
from datetime import datetime

SAVE_DIR = "paper_trades"
os.makedirs(SAVE_DIR, exist_ok=True)

TODAY = datetime.now().strftime("%Y-%m-%d")
CSV_PATH = f"{SAVE_DIR}/niftybank_paper_{TODAY}.csv"

pd.DataFrame(paper_trades).to_csv(CSV_PATH, index=False)
pd.DataFrame(paper_trades).to_csv(CSV_PATH, index=False)

pd.DataFrame(paper_trades).to_csv(CSV_PATH, index=False)
print(f"Trades saved to {CSV_PATH}")


Trades saved to paper_trades/niftybank_paper_2025-12-25.csv
